In [1]:
!pip install -q \
transformers \
accelerate \
datasets \
peft \
scikit-learn \
sentencepiece \
bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 52.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login
import os

HF_TOKEN = userdata.get('HF_TOKEN')

login(HF_TOKEN)

In [3]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name())
    print(
        torch.cuda.get_device_properties(0).total_memory / 1e9,
        "GB"
    )

CUDA: True
NVIDIA L4
23.65915136 GB


In [4]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModel
)


MODEL="Qwen/Qwen3.5-4B-Base"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    token=HF_TOKEN
)


model = AutoModel.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)


model.eval()

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Qwen3_5Model(
  (visual): Qwen3_5VisionModel(
    (patch_embed): Qwen3_5VisionPatchEmbed(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (pos_embed): Embedding(2304, 1024)
    (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-23): 24 x Qwen3_5VisionBlock(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): Qwen3_5VisionAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (mlp): Qwen3_5VisionMLP(
          (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (act_fn): GELUTanh()
        )
      )
    )
    (merger): Qwen3_5VisionPatchMerger(
      (norm): LayerNorm((1024,), eps=1e-06, ele

In [5]:
def model_size(model):

    params=sum(
        p.numel()
        for p in model.parameters()
    )

    return params/1e9


print(
    "Backbone params:",
    model_size(model),
    "B"
)

Backbone params: 4.539265536 B


In [6]:
from transformers import AutoModelForCausalLM


full_model=AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)


print(
    "Full params:",
    model_size(full_model),
    "B"
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Full params: 4.205751296 B


In [7]:
examples=[

"""
State:
Employee downloaded invoice.zip
from unknown sender.

Question:
Is this phishing?

Choices:
A normal activity
B malicious phishing
""",


"""
State:
Production database is offline.

Question:
Severity?

Choices:
Low
Medium
Critical
"""

]

In [8]:
def get_hidden(text):

    inputs=tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        out=model(
            **inputs,
            output_hidden_states=True
        )


    return out.last_hidden_state

In [9]:
def pool(hidden):

    return {

    "last":
        hidden[:,-1,:],

    "mean":
        hidden.mean(dim=1),

    "max":
        hidden.max(dim=1).values

    }

In [10]:
for text in examples:

    h=get_hidden(text)

    print(
        h.shape
    )

    pools=pool(h)

    for k,v in pools.items():

        print(
            k,
            v.shape
        )

torch.Size([1, 33, 2560])
last torch.Size([1, 2560])
mean torch.Size([1, 2560])
max torch.Size([1, 2560])
torch.Size([1, 25, 2560])
last torch.Size([1, 2560])
mean torch.Size([1, 2560])
max torch.Size([1, 2560])


In [11]:
import torch.nn as nn


class DecisionHead(nn.Module):

    def __init__(self, hidden):

        super().__init__()

        self.net=nn.Sequential(

            nn.Linear(hidden,512),

            nn.GELU(),

            nn.Dropout(.1),

            nn.Linear(512,2)

        )


    def forward(self,x):

        return self.net(x)

In [12]:
head=DecisionHead(2560).cuda()

In [13]:
dataset=[

{
"text":
"Server received 1000 failed SSH attempts",

"label":1
},


{
"text":
"User logged into office VPN",

"label":0
}

]

In [14]:
for p in model.parameters():

    p.requires_grad=False

In [15]:
optimizer=torch.optim.AdamW(
    head.parameters(),
    lr=1e-3
)


loss_fn=nn.CrossEntropyLoss()

In [16]:
for epoch in range(10):

    total=0


    for item in dataset:

        h=get_hidden(
            item["text"]
        )

        x=h.mean(dim=1).float()


        logits=head(x.cuda())


        loss=loss_fn(
            logits,
            torch.tensor(
                [item["label"]]
            ).cuda()
        )


        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        total+=loss.item()


    print(epoch,total)

0 9.590319871902466
1 2.467608533152088e-05
2 0.03146650269627571
3 0.021457545459270477
4 0.0006640136707574129
5 0.00019703354337252676
6 0.012482375837862492
7 0.0005998004344291985
8 0.0
9 0.0


In [17]:
import time


prompt="""
Determine if this security event is malicious:
Failed SSH logins from unknown IP.
"""

In [18]:
start=time.time()

out=full_model.generate(
    **tokenizer(
        prompt,
        return_tensors="pt"
    ).to(full_model.device),
    max_new_tokens=100
)

print(
    time.time()-start
)

7.480607748031616


In [19]:
start=time.time()

h=get_hidden(prompt)

x=h.mean(dim=1).float()

with torch.no_grad():

    result=head(
        x.cuda()
    )


print(
    time.time()-start
)

0.2616415023803711


In [20]:
if torch.cuda.is_available():

    print(
    torch.cuda.memory_summary()
    )

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  16717 MiB |  16790 MiB | 100074 MiB |  83356 MiB |
|       from large pool |  16696 MiB |  16752 MiB |  68424 MiB |  51728 MiB |
|       from small pool |     20 MiB |     41 MiB |  31649 MiB |  31628 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  16717 MiB |  16790 MiB | 100074 MiB |  83356 MiB |
|       from large pool |  16696 MiB |  16752 MiB |  68424 MiB |

In [25]:
import json


text_config = model.config.text_config


summary = {

    "model": MODEL,

    "hidden_size":
        text_config.hidden_size,

    "layers":
        text_config.num_hidden_layers,

    "vocab_size":
        text_config.vocab_size,

    "architecture":
        model.config.architectures,

    "pooling_test":
        [
            "last",
            "mean",
            "max"
        ],

    "removed_generation":
        True

}


with open(
    "opendecision_phase2_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print(json.dumps(summary, indent=2))

{
  "model": "Qwen/Qwen3.5-4B-Base",
  "hidden_size": 2560,
  "layers": 32,
  "vocab_size": 248320,
  "architecture": [
    "Qwen3_5ForConditionalGeneration"
  ],
  "pooling_test": [
    "last",
    "mean",
    "max"
  ],
  "removed_generation": true
}


In [26]:
summary["model_type"] = text_config.model_type

summary["attention_type"] = getattr(
    text_config,
    "attention_type",
    None
)

summary["head_dim"] = getattr(
    text_config,
    "head_dim",
    None
)

summary["num_attention_heads"] = getattr(
    text_config,
    "num_attention_heads",
    None
)

summary["num_key_value_heads"] = getattr(
    text_config,
    "num_key_value_heads",
    None
)